In [0]:
%python
# ==============================================================================
# PERFORMANCE OPTIMIZATION: LIQUID CLUSTERING & OPTIMIZE COMPACTION
# USE CASE: TALENT ASSET & RISK AUDIT GOVERNANCE
# ==============================================================================

# --- 1. MEMBUAT TABEL DENGAN LIQUID CLUSTERING ---
# Pada skenario Risk & Talent, kueri paling sering memfilter berdasarkan:
# - dept_code (analisis tingkat divisi)
# - standardized_risk_score (penyaringan ambang batas anomali risiko)
# - employee_id (audit riwayat individu)

spark.sql("""
CREATE OR REPLACE TABLE training_medallion.02_silver.silver_talent_risk_audits_clustered
CLUSTER BY (dept_code, standardized_risk_score)
AS SELECT * FROM training_medallion.02_silver.silver_talent_risk_audits
""")
print("Tabel Silver dengan Liquid Clustering berhasil dibuat.")


# --- 2. VERIFIKASI METADATA CLUSTERING PADA TABEL ---
# Memastikan Unity Catalog mengenali konfigurasi Liquid Clustering
display(spark.sql("DESCRIBE DETAIL training_medallion.02_silver.silver_talent_risk_audits_clustered"))


# --- 3. MENJALANKAN OPTIMIZE DENGAN LIQUID CLUSTERING ---
# Perintah OPTIMIZE pada tabel berklaster otomatis melakukan pemadatan (compaction)
# dan penyusunan ulang data sesuai kolom CLUSTER BY tanpa perlu klausa ZORDER BY
print("\nMenjalankan OPTIMIZE pada tabel Silver (Liquid Clustering)...")
optimize_silver_result = spark.sql("OPTIMIZE training_medallion.02_silver.silver_talent_risk_audits_clustered")
display(optimize_silver_result)


# --- 4. MERUBAH / EVOLUSI KOLOM CLUSTER (ALTER CLUSTERING) ---
# Jika pola query tim Risk Management berubah (misal: perlu memfilter audit_id atau employee_id),
# kolom klaster dapat langsung diubah tanpa perlu membuat ulang tabel
spark.sql("""
ALTER TABLE training_medallion.02_silver.silver_talent_risk_audits_clustered
CLUSTER BY (employee_id, standardized_risk_score)
""")
print("Kunci Liquid Clustering berhasil diperbarui ke (employee_id, standardized_risk_score).")


# --- 5. OPTIMASI TABEL GOLD (STANDARD COMPACTION & BIN-PACKING) ---
# Untuk tabel Gold agregat kecil hingga menengah, OPTIMIZE memadatkan file-file kecil
# menjadi ukuran standar (~1 GB) untuk mempercepat performa dashboard BI
print("\nMenjalankan OPTIMIZE pemadatan file pada Gold Summary Table...")
optimize_gold_result = spark.sql("OPTIMIZE training_medallion.03_gold.gold_department_risk_summary")
display(optimize_gold_result)


# --- 6. VACUUM: PEMBERSIHAN FILE HISTORIS YANG TIDAK TERPAKAI ---
# Membersihkan snapshot file lama hasil penataan ulang OPTIMIZE (retensi default 168 jam / 7 hari)
# Gunakan DRY RUN terlebih dahulu untuk memeriksa file yang memenuhi syarat pembersihan
display(spark.sql("VACUUM training_medallion.02_silver.silver_talent_risk_audits_clustered DRY RUN"))

Tabel Silver dengan Liquid Clustering berhasil dibuat.


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,33253912-ce19-433d-8966-99fd6e6c2e2c,training_medallion.02_silver.silver_talent_risk_audits_clustered,null,,2026-10-29T15:10:30.048Z,2026-10-29T15:10:32.000Z,List(),"List(dept_code, standardized_risk_score)",1,4067,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, io.unitycatalog.tableId -> ff33c42f-545b-4c1d-b044-fdb1b11375c2, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-f0a50b51-3f9f-4784-841c-e83136e5c2d8, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-b578ea5d-c509-488b-9b19-bc903ed23f8d)",3,7,"List(appendOnly, clustering, deletionVectors, domainMetadata, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false



Menjalankan OPTIMIZE pada tabel Silver (Liquid Clustering)...


path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 0, false, 0, 0, 1793286635463, 1793286642214, 8, 0, null, List(0, 0), null, 12, 12, 0, 0, List(4067, true, false, false, null, null, null, null, 0, 0, 0, 0, 1, 4067, 4067, null, log, 16777216, 67108864, 4, 0, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(174, 115, 0, 0, 0, 0, 2475), 2, 1, 5, sizeAware, defaultSplitStrategy, null, false, 0, null, false, 0, 0, 0, null, null, null, -1.0, n/a, 0, 0, false, false, -1.0, n/a), null, 0)"
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1793286642301, 1793286644021, 8, 0, null, List(0, 0), null, 12, 12, 0, 0, List(4067, false, false, false, null, null, null, post-optimize-compaction, 0, 0, 0, 0, 0, 0, 0, null, null, 33554432, 67108864, 0, 0, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(0, 0, 344, 0, 0, 0, 0), 15, 1, 1, null, null, null, false, 0, null, false, 0, 0, 0, null, null, null, -1.0, n/a, 0, 0, false, false, -1.0, n/a), null, 0)"


Kunci Liquid Clustering berhasil diperbarui ke (employee_id, standardized_risk_score).

Menjalankan OPTIMIZE pemadatan file pada Gold Summary Table...


path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1793286647598, 1793286648497, 8, 0, null, List(0, 0), null, 7, 7, 0, 0, null, null, 0)"


path


In [0]:
EXPLAIN FORMATTED
SELECT * 
FROM training_medallion.02_silver.silver_talent_risk_audits_clustered
WHERE employee_id = 101 AND standardized_risk_score >= 80.0;

plan
"== Physical Plan == PhotonResultStage (3) +- PhotonColumnarToRow (2) +- PhotonScan parquet training_medallion.02_silver.silver_talent_risk_audits_clustered (1) (1) PhotonScan parquet training_medallion.02_silver.silver_talent_risk_audits_clustered Output [12]: [audit_id#19050, employee_id#19051L, dept_code#19052, department_name#19053, standardized_risk_score#19054, max_risk_threshold#19055, compliance_level#19056, talent_risk_status#19057, is_escalated#19058, flag_reason#19059, ingestion_timestamp#19060, silver_processed_at#19061] DictionaryFilters: [(standardized_risk_score#19054 >= 80.0), (employee_id#19051L = 101)] Location: PreparedDeltaFileIndex [s3://dbstorage-prod-xev11tevgt/uc/80e29d41-51c7-4ca7-8103-a3bf845cf586/1cbaf109-f551-4000-b508-32dde4888498/__unitystorage/catalogs/38e5d4b1-ab96-4dfd-ac04-d355485dc993/tables/ff33c42f-545b-4c1d-b044-fdb1b11375c2] ReadSchema: struct RequiredDataFilters: [isnotnull(employee_id#19051L), isnotnull(standardized_risk_score#19054), (employee_id#19051L = 101), (standardized_risk_score#19054 >= 80.0)] (2) PhotonColumnarToRow Input [12]: [audit_id#19050, employee_id#19051L, dept_code#19052, department_name#19053, standardized_risk_score#19054, max_risk_threshold#19055, compliance_level#19056, talent_risk_status#19057, is_escalated#19058, flag_reason#19059, ingestion_timestamp#19060, silver_processed_at#19061] (3) PhotonResultStage Input [12]: [audit_id#19050, employee_id#19051L, dept_code#19052, department_name#19053, standardized_risk_score#19054, max_risk_threshold#19055, compliance_level#19056, talent_risk_status#19057, is_escalated#19058, flag_reason#19059, ingestion_timestamp#19060, silver_processed_at#19061] == Photon Explanation == The query is fully supported by Photon. == Optimizer Statistics (table names per statistics state) == missing = partial = full = silver_talent_risk_audits_clustered"


In [0]:
%python
# Execute the query to collect runtime scan metrics
df_result = spark.sql("""
SELECT * 
FROM training_medallion.02_silver.silver_talent_risk_audits_clustered
WHERE employee_id = 101 AND standardized_risk_score >= 80.0
""")

display(df_result)

audit_id,employee_id,dept_code,department_name,standardized_risk_score,max_risk_threshold,compliance_level,talent_risk_status,is_escalated,flag_reason,ingestion_timestamp,silver_processed_at
AUD-101,101,FIN,Finance & Treasury,88.0,70.0,Strict,HIGH RISK EXCEEDED,true,Excessive Access,2026-10-25T17:49:25.045Z,2026-10-25T17:49:56.555Z
